# **Pseudo Kaggle — local stand-in for the leaderboard**

Kaggle's validation/test rows are private and submissions are limited, so this notebook mimics the scoring locally:

1. Train the two fixed detectors on a synthetic CSV: **ECOD**, and **IForest** with 10 seeds × 500 trees, averaged.
2. Let them score real rows from `train.csv`, where we know which rows are attacks.
3. Report AUPRC per detector and **S = (AUPRC_ECOD + AUPRC_IForest) / 2**.

The organizers' preprocessing is unknown. We use one-hot categoricals + StandardScaler, the same as the EDA notebooks. ECOD and IForest barely care about per-column scaling, and ordinal encoding gave nearly the same numbers.

**Two eval modes:**

| mode | eval rows | use it for |
|---|---|---|
| `all` | all train normals + **all** train anomalies | Matches Kaggle well. Calibration: VAE baseline gave NSL 0.929 vs Kaggle 0.930, UNSW 0.496 vs Kaggle 0.447. **Optimistic** for any file whose filter classifier saw these anomalies. |
| `holdout` | all train normals + anomaly **half B** only | Fair comparison of anomaly-aware methods. Pair it with `clf_holdout=True` in `VAE.ipynb`, which only lets the filter see half A. Scores are lower than Kaggle because there are fewer anomalies, so compare files with each other, not with Kaggle. |

Half A/B split: shuffle the anomaly rows with `random_state=0`; the first half is A and the second half is B. It must match `VAE.ipynb`.

**Caveat:** train anomalies are only `A_seen` types. Kaggle validation adds `A_val` and the hidden test adds `A_test`, which pseudo kaggle can't see. Gains that come from targeting seen attack types may not fully transfer.

---
**Config**

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from pyod.models.ecod import ECOD
from pyod.models.iforest import IForest
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

DATASET = "UNSW"   # "NSL" | "UNSW"
MODE = "holdout"   # "all" | "holdout"
N_SEEDS = 10       # IForest seeds (grading spec: 10 x 500 trees)

ROOT = Path("/Users/han./Desktop/CS5344")
CONFIGS = {
    "NSL": dict(path=ROOT / "data/raw/NSL-KDD_train.csv",
                cat_cols=["protocol_type", "service", "flag"], N=40_000),
    "UNSW": dict(path=ROOT / "data/raw/UNSW-NB15_train.csv",
                 cat_cols=["proto", "service", "state"], N=70_000),
}
cfg = CONFIGS[DATASET]
LABEL = "is_anomaly"

# synthetic CSVs to score: every file for this dataset in data/processed (edit to pick specific ones)
FILES = sorted((ROOT / "data/processed").glob(f"synthetic_{DATASET}*.csv"))
[f.name for f in FILES]

---
**Eval set + scoring function**

In [ ]:
df = pd.read_csv(cfg["path"])
cat_cols = cfg["cat_cols"]
feature_cols = [c for c in df.columns if c != LABEL]
normal_df = df[df[LABEL] == 0]

anom = df[df[LABEL] == 1].sample(frac=1, random_state=0)   # same split as VAE.ipynb
anom_B = anom.iloc[len(anom) // 2:]
eval_df = df if MODE == "all" else pd.concat([normal_df, anom_B], ignore_index=True)
print(f"{DATASET} mode={MODE}: eval rows={len(eval_df)}, anomaly share={eval_df[LABEL].mean():.3f}")

def pseudo_kaggle(synth, eval_df=eval_df, n_seeds=N_SEEDS):
    """Train ECOD + IForest(n_seeds x 500) on `synth`, score `eval_df`, return AUPRCs and S."""
    train = pd.get_dummies(synth[feature_cols], columns=cat_cols, dtype=float)
    test = pd.get_dummies(eval_df[feature_cols], columns=cat_cols, dtype=float)
    test = test.reindex(columns=train.columns, fill_value=0.0)  # categories unseen in synth -> all-zero
    scaler = StandardScaler().fit(train)
    A, B = scaler.transform(train), scaler.transform(test)
    y = eval_df[LABEL].to_numpy()
    ecod = average_precision_score(y, ECOD().fit(A).decision_function(B))
    iforest = np.mean([average_precision_score(y, IForest(n_estimators=500, random_state=s).fit(A).decision_function(B))
                       for s in range(n_seeds)])
    return dict(ECOD=ecod, IForest=iforest, S=(ecod + iforest) / 2)

---
**Reference: real normal rows resampled to N**

This is the score of a 'perfect fidelity' submission. A method that only imitates the normal distribution can't do much better than this.

In [ ]:
rng = np.random.default_rng(0)
real_resampled = normal_df.iloc[rng.integers(0, len(normal_df), cfg["N"])]
results = {"[ref] real normals resampled": pseudo_kaggle(real_resampled)}
results["[ref] real normals resampled"]

---
**Score every synthetic file**

In [ ]:
for f in FILES:
    synth = pd.read_csv(f)
    assert len(synth) == cfg["N"], f"{f.name}: {len(synth)} rows, expected {cfg['N']}"
    results[f.name] = pseudo_kaggle(synth)
    print(f"{f.name:45s} " + " ".join(f"{k}={v:.4f}" for k, v in results[f.name].items()))

pd.DataFrame(results).T.sort_values("S", ascending=False).round(4)